# Smartnode customer-support voice triage POC

This notebook downloads a few English customer-support speech samples from PolyAI's **MInDS-14** dataset, transcribes them locally with `faster-whisper`, and assigns each transcript a transparent `Closed`, `Open`, or `Urgent` triage label. It also checks the rules against representative Smartnode-style examples.

## Project objective and end-to-end flow

The goal is to demonstrate a small, reproducible proof of concept for reducing manual call handling: convert an audio recording into text, retain the text for agent review, and attach a first-pass routing category with a reason. The processing flow is **public sample lookup → audio download → local speech recognition → rule-based triage → CSV output**. The transcription step uses an English speech-recognition model; the triage step is separate from ASR and applies explicit phrase rules to the transcript. No customer database, Smartnode credentials, or external inference API is used.

## Cell-by-cell run guide

Run the notebook from top to bottom, one cell at a time (or use **Run all**):

1. Read this overview and the assumptions before using any data.
2. Install the notebook's Python libraries. In Colab this may prompt for a kernel restart; if so, reconnect and continue at the imports cell.
3. Import libraries and set the sample count, model name, and output folders.
4. Ask the Hugging Face dataset viewer for three English examples and download each audio file. The code checks HTTP errors, missing audio, and empty downloads rather than silently continuing.
5. Load `tiny.en` once and transcribe every downloaded clip locally. The first run downloads model weights and may take a few minutes on CPU.
6. Define and run small rule checks covering all three labels, urgency precedence, negated safety wording, and the conservative Open fallback.
7. Triage the real sample transcripts and save the complete transcript, reference transcript, category, reason, and source intent to `poc_outputs/transcribed_calls.csv`.
8. Review the final checklist before applying this pattern to real customer calls.

Expected outputs are the cell tables, five passing triage checks, and a CSV in the notebook's current working directory. Re-running cells reuses existing audio files by downloading them again; the sample set stays intentionally small. An internet connection is needed for sample retrieval and the first model download. Keep the notebook's working directory private if you replace the public data with customer recordings.

**Important:** MInDS-14 is banking-intent speech—not Smartnode calls. It exercises the audio pipeline only. The three-way rule classifier is an illustrative baseline, not a learned or validated model. Do not use this POC to make unattended safety decisions or upload real customer recordings.

Dataset card/license: [PolyAI/MInDS-14](https://huggingface.co/datasets/PolyAI/minds14) (the card states CC-BY-4.0). Audio/English transcript sample API: [Hugging Face dataset viewer](https://datasets-server.huggingface.co/rows?dataset=PolyAI%2Fminds14&config=en-US&split=train&offset=0&length=3).

In [ ]:
%pip -q install "faster-whisper>=1.0,<2" "av>=11,<19" "requests>=2.31,<3" "pandas>=2,<4"

In [ ]:
from pathlib import Path
import re

import pandas as pd
import requests
from faster_whisper import WhisperModel

DATASET = "PolyAI/minds14"
CONFIG = "en-US"
SPLIT = "train"
SAMPLE_COUNT = 3  # Keep the demo download small.
MODEL_NAME = "tiny.en"  # Local CPU-friendly English model.
AUDIO_DIR = Path("poc_data")
OUTPUT_DIR = Path("poc_outputs")
AUDIO_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Dataset: {DATASET} ({CONFIG}/{SPLIT}), first {SAMPLE_COUNT} rows")

In [ ]:
# Query only a few rows; this avoids downloading the full ~0.6 GB dataset.
rows_url = "https://datasets-server.huggingface.co/rows"
response = requests.get(
    rows_url,
    params={"dataset": DATASET, "config": CONFIG, "split": SPLIT,
            "offset": 0, "length": SAMPLE_COUNT},
    timeout=60,
)
response.raise_for_status()
rows_payload = response.json()
samples = [item["row"] for item in rows_payload["rows"]]
if not samples:
    raise RuntimeError("The dataset viewer returned no sample rows.")

def download_audio(sample, index):
    audio_info = sample.get("audio")
    if not audio_info or not audio_info[0].get("src"):
        raise ValueError(f"Sample {index} has no downloadable audio URL.")
    audio_url = audio_info[0]["src"]
    audio_response = requests.get(audio_url, timeout=120)
    audio_response.raise_for_status()
    audio_path = AUDIO_DIR / f"sample_{index:02d}.wav"
    audio_path.write_bytes(audio_response.content)
    if audio_path.stat().st_size == 0:
        raise RuntimeError(f"Downloaded audio file is empty: {audio_path}")
    return audio_path

for index, sample in enumerate(samples):
    sample["local_audio_path"] = str(download_audio(sample, index))

print(f"Downloaded {len(samples)} audio files to {AUDIO_DIR.resolve()}")
pd.DataFrame([
    {"file": s["path"], "reference_transcript": s["english_transcription"],
     "intent": s["intent_class"]}
    for s in samples
])

## Local transcription

The first run downloads the small Whisper model from its model hub. Audio stays in this notebook environment; it is not sent to an inference API. `device="cpu"` and `compute_type="int8"` keep the demo compatible with ordinary CPU runtimes. `vad_filter=True` skips non-speech sections, while `language="en"` is appropriate because the selected dataset configuration is English (US). Transcripts are compared with the dataset's reference text as a basic pipeline sanity check, not as a formal word-error-rate evaluation.

In [ ]:
whisper = WhisperModel(MODEL_NAME, device="cpu", compute_type="int8")

def transcribe_file(audio_path):
    segments, info = whisper.transcribe(
        str(audio_path), language="en", vad_filter=True, beam_size=3
    )
    text = " ".join(segment.text.strip() for segment in segments).strip()
    if not text:
        raise RuntimeError(f"Whisper produced an empty transcript for {audio_path}")
    return text, info.language

for sample in samples:
    sample["transcript"], sample["detected_language"] = transcribe_file(sample["local_audio_path"])

pd.DataFrame([
    {"file": s["path"], "reference": s["english_transcription"],
     "transcript": s["transcript"], "language": s["detected_language"]}
    for s in samples
])

## Explainable triage baseline

Operational definitions for this demo:

- **Urgent:** an explicit immediate safety risk or critical service interruption; highest priority.
- **Closed:** the customer explicitly confirms the issue is resolved.
- **Open:** action or resolution remains pending; also the conservative default when evidence is insufficient.

The keyword lists are examples to review with Smartnode agents and safety/operations owners. They are not exhaustive, and a production system should preserve the matched reason, allow agent overrides, and be evaluated on representative, consented, appropriately redacted Smartnode calls.

The decision order is intentional: an urgent cue wins even when the same transcript also contains a resolution cue; otherwise an explicit closure phrase is Closed; all other valid input is Open. A small context check avoids treating a safety phrase immediately preceded by common negation (for example, “no smoke”) as an urgent match. This context check is not a general natural-language understanding system—complex wording still requires agent review. Each output includes both the category and a short reason so an agent can understand why the rule fired.

In [ ]:
URGENT_PATTERN = re.compile(
    r"\b(smoke|smoking|fire|sparks?|burning smell|electric shock|"
    r"exposed wires?|gas leak|carbon monoxide|flooding|water leak|"
    r"emergency|no power|total outage|system is down|not working at all)\b",
    re.IGNORECASE,
)
CLOSED_PATTERN = re.compile(
    r"\b(resolved|fixed|working now|back to normal|that solved it|"
    r"issue is gone|all set|nothing else|no further help)\b",
    re.IGNORECASE,
)
NEGATION_PATTERN = re.compile(r"\b(?:no|not|without|never|isn't|aren't)\s+(?:\w+\s+){0,2}$", re.IGNORECASE)

def classify_contact(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("A non-empty transcript is required for triage.")
    urgent_match = None
    for match in URGENT_PATTERN.finditer(text):
        prefix = text[max(0, match.start() - 32):match.start()]
        if not NEGATION_PATTERN.search(prefix):
            urgent_match = match.group(0)
            break
    if urgent_match:
        return {"category": "Urgent", "reason": f"Urgent risk/service phrase: '{urgent_match}'"}

    closed_match = CLOSED_PATTERN.search(text)
    if closed_match:
        return {"category": "Closed", "reason": f"Explicit resolution cue: '{closed_match.group(0)}'"}

    return {"category": "Open", "reason": "No explicit resolution or urgent trigger; follow-up/review may be needed."}

# Checks for business meaning, urgency precedence, negation, and conservative default.
triage_examples = [
    ("The inverter is smoking and there are sparks near the battery.", "Urgent"),
    ("There is no smoke and no sparks; the app asks me to reconnect.", "Open"),
    ("Please check the installation; it still is not working.", "Open"),
    ("Thanks, the technician fixed it and the system is working now.", "Closed"),
    ("The power is back to normal, but there is a burning smell.", "Urgent"),
]
for text, expected in triage_examples:
    actual = classify_contact(text)["category"]
    assert actual == expected, f"Expected {expected}, got {actual}: {text}"

print("All triage example checks passed.")
pd.DataFrame([
    {"example": text, **classify_contact(text), "expected": expected}
    for text, expected in triage_examples
])

In [ ]:
results = []
for sample in samples:
    triage = classify_contact(sample["transcript"])
    results.append({
        "source_file": sample["path"],
        "reference_transcript": sample["english_transcription"],
        "transcript": sample["transcript"],
        "category": triage["category"],
        "reason": triage["reason"],
        "dataset_intent_not_triage_label": sample["intent_class"],
    })

results_df = pd.DataFrame(results)
output_path = OUTPUT_DIR / "transcribed_calls.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} demo results to {output_path.resolve()}")
results_df

## What to validate before a real pilot

1. Agree on exact Closed/Open/Urgent definitions and safety escalation SLAs with customer support, dealer operations, and safety owners.
2. Build a consented, redacted, representative Smartnode call set and have agents label outcomes; keep customer intent separate from call-resolution status.
3. Measure transcription error, triage precision/recall (especially urgent recall), agent override rate, routing time, after-call work, repeat contact rate, and CSAT by language/product/dealer.
4. Review privacy, retention, access, model licensing, and human override controls. Do not auto-close a case based only on keywords.

The example scorecard targets in the companion slide are hypotheses to test, not results from this notebook. This three-row public sample is too small and is not labeled for call resolution or urgency, so it cannot establish classification accuracy, time savings, or customer impact. A meaningful evaluation needs an agreed labeling guide, a representative held-out set, and separate reporting for urgent recall and false alarms.